In [1]:
push!(LOAD_PATH, "./src")
include("src/CNN.jl")
using Random
using MLDatasets

In [2]:
function onehot(target, classes=0:9)
    num_classes = length(classes)
    encoded = zeros(Float32, num_classes)
    idx = target - first(classes) + 1
    encoded[idx] = 1.0f0
    return encoded
end

function loader(data; batchsize::Int=32)
    x_all = Float32.(reshape(data.features, 28, 28, 1, :))
    targets = data.targets
    
    n_samples = size(x_all, 4)
    indices = randperm(n_samples)
    
    return (
        begin
            batch_idx = indices[i:min(i + batchsize - 1, n_samples)]
            
            x_batch = x_all[:, :, :, batch_idx]
            y_batch = hcat([onehot(targets[idx], 0:9) for idx in batch_idx]...)
            
            (x_batch, y_batch)
        end
        for i in 1:batchsize:n_samples
    )
end

loader (generic function with 1 method)

In [3]:
import .CNN: chain, conv2d, MaxPool, Flatten, dense, relu, Dropout, softmax, GraphNode, CrossEntropy, graph, train_step!, loss_and_accuracy_clean
using MLDatasets
using Random

# Load data
train_mnist = FashionMNIST(split=:train)
test_mnist  = FashionMNIST(split=:test)

train_data = (
    features = train_mnist.features,
    targets  = train_mnist.targets,
)

test_data = (
    features = test_mnist.features,
    targets  = test_mnist.targets,
)

# Define architecture
net = chain((
  conv2d((3, 3), 1 => 6, pad=1, bias=false),
  MaxPool((2, 2)),
  conv2d((3, 3), 6 => 16, pad=1, bias=false),
  MaxPool((2, 2)),
  Flatten(),
  dense(784 => 84, relu),
  Dropout(0.4),
  dense(84 => 10),
  softmax()
))

input_tensor = GraphNode(zeros(Float32, 28, 28, 1))
target_tensor = GraphNode(zeros(Float32, 10))
output_node = net(input_tensor)
loss_node = CrossEntropy()(output_node, target_tensor)
model_graph = graph(loss_node)

settings = (; eta = 0.01f0, epochs = 3, batchsize = 10)
accuracy = zeros(settings.epochs, 2)
train_log = []

# # Training loop
# for epoch in 1:settings.epochs
#     @time for (x_batch, y_batch) in loader(train_data, batchsize=settings.batchsize)
#         train_step!(model_graph, input_tensor, target_tensor, x_batch, y_batch, settings.eta)
#     end
    
#     train_stats = loss_and_accuracy_clean(model_graph, input_tensor, output_node, loss_node, target_tensor, loader(train_data, batchsize=1000))
#     test_stats  = loss_and_accuracy_clean(model_graph, input_tensor, output_node, loss_node, target_tensor, loader(test_data, batchsize=1000))
    
#     println("[Epoka $epoch] Dokładność (Train): $(train_stats.acc)% | (Test): $(test_stats.acc)%")
#     flush(stdout)
    
#     push!(train_log, (; epoch, train_stats..., test_stats...))
#     accuracy[epoch, 1] = train_stats.acc
#     accuracy[epoch, 2] = test_stats.acc
# end

Any[]

In [ ]:
import .CNN: chain, conv2d, MaxPool, Flatten, dense, relu, Dropout, softmax, GraphNode, CrossEntropy, graph, train_step!, loss_and_accuracy_clean
using MLDatasets
using Random

# Load data
train_mnist = FashionMNIST(split=:train)
test_mnist  = FashionMNIST(split=:test)

train_data = (
    features = train_mnist.features,
    targets  = train_mnist.targets,
)

test_data = (
    features = test_mnist.features,
    targets  = test_mnist.targets,
)

net = chain((
  conv2d((3, 3), 1 => 6, pad=1, bias=false),
  MaxPool((2, 2)),
  conv2d((3, 3), 6 => 16, pad=1, bias=false),
  MaxPool((2, 2)),
  Flatten(),
  dense(784 => 84, relu),
  Dropout(0.4),
  dense(84 => 10),
  softmax()
))

input_tensor = GraphNode(zeros(Float32, 28, 28, 1))
target_tensor = GraphNode(zeros(Float32, 10))
output_node = net(input_tensor)
loss_node = CrossEntropy()(output_node, target_tensor)
model_graph = graph(loss_node)

settings = (; eta = 0.01f0, epochs = 3, batchsize = 10)
accuracy = zeros(settings.epochs, 2)

total_train_time = 0.0
total_allocs_bytes = 0
total_allocs_count = 0 

# Training loop
for epoch in 1:settings.epochs
    stats = @timed begin
        for (x_batch, y_batch) in loader(train_data, batchsize=settings.batchsize)
            train_step!(model_graph, input_tensor, target_tensor, x_batch, y_batch, settings.eta)
        end
    end
    
    epoch_allocs_count = stats.gcstats.poolalloc + stats.gcstats.bigalloc + stats.gcstats.malloc + stats.gcstats.realloc
    
    total_train_time += stats.time
    total_allocs_bytes += stats.bytes
    total_allocs_count += epoch_allocs_count
    
    train_stats = loss_and_accuracy_clean(model_graph, input_tensor, output_node, loss_node, target_tensor, loader(train_data, batchsize=1000))
    test_stats  = loss_and_accuracy_clean(model_graph, input_tensor, output_node, loss_node, target_tensor, loader(test_data, batchsize=1000))
    
    epoch_time_s = round(stats.time, digits=2)
    epoch_alloc_mb = round(stats.bytes / 1024^2, digits=2)
    println("[Epoka $epoch] Czas: $(epoch_time_s)s | Alokacje: $(epoch_allocs_count) ($(epoch_alloc_mb) MB) | Dokładność (Train): $(train_stats.acc)% | (Test): $(test_stats.acc)%")
    flush(stdout)
    
    accuracy[epoch, 1] = train_stats.acc
    accuracy[epoch, 2] = test_stats.acc
end

println("\n--- PODSUMOWANIE ---")
println("Czas (s);Alokacje;Pamięć (MB);Train Acc (%);Test Acc (%)")

to_pl(val) = replace(string(round(val, digits=2)), "." => ",")

total_mem_mb = total_allocs_bytes / 1024^2
final_train_acc = accuracy[end, 1]
final_test_acc = accuracy[end, 2]

row = "$(to_pl(total_train_time));$(total_allocs_count);$(to_pl(total_mem_mb));$(to_pl(final_train_acc));$(to_pl(final_test_acc))"
println(row)

filename = "wyniki_eksperymentow.txt"
file_exists = isfile(filename)

open(filename, "a") do file
    if !file_exists
        write(file, "Czas (s);Alokacje;Pamięć (MB);Train Acc (%);Test Acc (%)\n")
    end
    write(file, row * "\n")
end

[Epoka 1] Czas: 6.53s | Alokacje: 10740739 (678.44 MB) | Dokładność (Train): 84.93% | (Test): 84.21%
[Epoka 2] Czas: 6.4s | Alokacje: 10739918 (678.4 MB) | Dokładność (Train): 86.32% | (Test): 85.22%
[Epoka 3] Czas: 6.43s | Alokacje: 10739918 (678.4 MB) | Dokładność (Train): 87.9% | (Test): 86.62%

--- PODSUMOWANIE ---
Czas (s);Alokacje;Pamięć (MB);Train Acc (%);Test Acc (%)
19,36;32220575;2035,24;87,9;86,62


34

# Testing performance

In [5]:
# import Pkg; Pkg.add("TimerOutputs")
using TimerOutputs

const to = TimerOutput()

────────────────────────────────────────────────────────────────────
                           Time                    Allocations      
                  ───────────────────────   ────────────────────────
Tot / % measured:      291ms /   0.0%           37.7MiB /   0.0%    

Section   ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────

In [6]:
function train_step!(graph, input_node, target_node, x_batch, y_batch, eta)
    @timeit to "1. zerograd" CNN.zerograd!(graph)
    batch_size = size(x_batch, 4)
    
    for i in 1:batch_size
        @timeit to "2. zero_act" CNN.zero_activations_grad!(graph)
        
        @timeit to "3. views" begin
            x_single = @view x_batch[:, :, :, i]
            y_single = @view y_batch[:, i]
        end
        
        @timeit to "4. forward" CNN.forward!(graph, input_node => x_single, target_node => y_single, train_mode=true)
        @timeit to "5. backward" CNN.backward!(graph)
    end
    
    @timeit to "6. optimize" CNN.optimize!(graph, eta / batch_size) 
end

train_step! (generic function with 1 method)

In [7]:
reset_timer!(to)

x_batch, y_batch = first(loader(train_data, batchsize=10))
train_step!(model_graph, input_tensor, target_tensor, x_batch, y_batch, settings.eta)

show(to)

────────────────────────────────────────────────────────────────────────
                               Time                    Allocations      
                      ───────────────────────   ────────────────────────
  Tot / % measured:        342ms /   0.5%            240MiB /   0.0%    

Section       ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────────
4. forward        10    858μs   51.9%  85.8μs   50.8KiB   61.3%  5.08KiB
5. backward       10    667μs   40.4%  66.7μs   26.9KiB   32.5%  2.69KiB
6. optimize        1   63.2μs    3.8%  63.2μs   3.98KiB    4.8%  3.98KiB
1. zerograd        1   52.4μs    3.2%  52.4μs   1.19KiB    1.4%  1.19KiB
2. zero_act       10   10.6μs    0.6%  1.06μs     0.00B    0.0%    0.00B
3. views          10    700ns    0.0%  70.0ns     0.00B    0.0%    0.00B
────────────────────────────────────────────────────────────────────────

In [8]:
# import Pkg
# Pkg.gc()
# Pkg.add("ProfileSVG")
using Profile
using ProfileSVG

ArgumentError: ArgumentError: Package ProfileSVG not found in current path.
- Run `import Pkg; Pkg.add("ProfileSVG")` to install the ProfileSVG package.

In [9]:
x_batch, y_batch = first(loader(train_data, batchsize=10))

train_step!(model_graph, input_tensor, target_tensor, x_batch, y_batch, settings.eta)

In [10]:
Profile.clear()

@profile for i in 1:20
    train_step!(model_graph, input_tensor, target_tensor, x_batch, y_batch, settings.eta)
end

In [11]:
ProfileSVG.view()
ProfileSVG.save("my_profile.svg")

UndefVarError: UndefVarError: `ProfileSVG` not defined in `Main`
Suggestion: check for spelling errors or missing imports.